In [1]:
import numpy as np
from qdk_chemistry.data import Element, Structure, MajoranaMapping
from qdk_chemistry.algorithms import create
from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)

# 1) Build H2 molecule and compute SCF energy and wavefunction
coords_ang = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 0.740848],
    ]
)
structure = Structure(coords_ang, [Element.H, Element.H])

scf = create("scf_solver")
scf_energy, wavefunction = scf.run(
    structure, charge=0, spin_multiplicity=1, basis_or_guess="sto-3g"
)

# 2) Electronic Hamiltonian
ham_constructor = create("hamiltonian_constructor")
hamiltonian = ham_constructor.run(wavefunction.get_orbitals())

# 3) Qubit Hamiltonian
mapper = create("qubit_mapper")
n_spin_orbitals = hamiltonian.get_orbitals().get_num_molecular_orbitals() * 2
qubit_hamiltonian = mapper.run(hamiltonian, MajoranaMapping.jordan_wigner(n_spin_orbitals))

# 4) Ground-state energy of the qubit Hamiltonian
solver = create("qubit_hamiltonian_solver")
qubit_energy, ground_state = solver.run(qubit_hamiltonian)
qubit_energy = qubit_energy

In [3]:
import qdk
from utils.qpe_utils import compute_evolution_time
from qdk_chemistry.data import AlgorithmRef
from qdk_chemistry.algorithms.state_preparation import identity_state_prep
from qdk_chemistry.utils.pauli_commutation import commutator_bound_first_order, commutator_bound_second_order

def compute_trotter_error(qubit_hamiltonian, evolution_time, trotter_order):
    """Compute the Trotter error bound for a single step (num_divisions=1)."""
    if trotter_order == 1:
        alpha = commutator_bound_first_order(qubit_hamiltonian)
        return alpha * evolution_time**2 / 2
    elif trotter_order == 2:
        alpha = commutator_bound_second_order(qubit_hamiltonian)
        return alpha * abs(evolution_time)**3 / 12
    else:
        raise ValueError(f"Unsupported trotter_order: {trotter_order}")

def run_trotter(m_precision, trotter_order):
    evolution_time = compute_evolution_time(qubit_hamiltonian, num_bits=m_precision)
    unitary_builder = AlgorithmRef("hamiltonian_unitary_builder", "trotter", time=evolution_time, order=trotter_order)
    circuit_mapper = AlgorithmRef("controlled_circuit_mapper", "pauli_sequence")
    circuit_builder = create(
        "qpe_circuit_builder", 
        "qdk_iterative",
        num_bits=m_precision,
        unitary_builder=unitary_builder,
        controlled_circuit_mapper=circuit_mapper,
    )
    state_prep = identity_state_prep(num_qubits=qubit_hamiltonian.num_qubits)
    iqpe_iter_circuits = circuit_builder.run(
        state_preparation=state_prep,
        qubit_hamiltonian=qubit_hamiltonian,
    )
    largest_circuit = iqpe_iter_circuits[0]
    result = largest_circuit.estimate()
    logical_estimate = result["logicalCounts"]
    trotter_error = compute_trotter_error(qubit_hamiltonian, evolution_time, trotter_order)

    return logical_estimate, trotter_error

# Collect results
results = []
for m_precision in [6, 8, 10]:
    for trotter_order in [1, 2]:
        logical_estimate, trotter_error = run_trotter(m_precision, trotter_order)
        results.append((m_precision, trotter_order, logical_estimate, trotter_error))

# Display as table
print(f"{'m_precision':<12} {'trotter_order':<14} {'numQubits':<11} {'rotationCount':<15} {'rotationDepth':<15} {'trotter_error':<15}")
print("-" * 83)
for m_prec, order, estimate, error in results:
    num_qubits = estimate["numQubits"]
    rot_count = estimate["rotationCount"]
    rot_depth = estimate["rotationDepth"]
    print(f"{m_prec:<12} {order:<14} {num_qubits:<11} {rot_count:<15} {rot_depth:<15} {error:<15.6e}")


m_precision  trotter_order  numQubits   rotationCount   rotationDepth   trotter_error  
-----------------------------------------------------------------------------------
6            1              5           928             800             2.316031e-01   
6            2              5           1792            1536            7.765906e-02   
8            1              5           3712            3200            2.265956e-01   
8            2              5           7168            6144            7.515412e-02   
10           1              5           14848           12800           2.278423e-01   
10           2              5           28672           24576           7.577523e-02   
